# Linear Regression — Titanic Dataset

This notebook is the first model in:

```text
05-Supervised-Learning
└── Regression
    └── Linear-Regression
```

We will use the final feature-engineered Titanic dataset:

```text
06_titanic_feature_engineered.csv
```

**Important:** Titanic's original `Survived` variable is binary, so predicting survival is a **classification** problem. To learn Linear Regression correctly, this notebook uses the continuous `Fare` feature as the regression target. Because the final preprocessing file contains a scaled `Fare`, the model predicts **scaled fare values**, not the original currency amount.

This is a learning exercise for understanding regression mechanics; it is not intended as a production fare-pricing model.

## 1. Learning Objectives

By the end of this notebook, you should understand:

- What regression is and when to use it
- Simple Linear Regression
- Multiple Linear Regression
- The equation of a linear model
- Mean Squared Error and the idea of a cost function
- How coefficients and intercept are interpreted
- Train/test splitting
- How Linear Regression is fitted with scikit-learn
- A small Linear Regression implementation from scratch
- Residuals and regression metrics
- Common assumptions and limitations of Linear Regression
- Why feature selection and leakage matter before modeling

## 2. What Is Regression?

**Regression** is supervised learning where the target variable is continuous or numerical.

Examples:

```text
House features  → House price
Hours studied   → Exam score
Engine features → Fuel consumption
Passenger data  → Fare
```

The model learns a relationship between input features `X` and a continuous target `y`.

For a single feature, Linear Regression assumes:

\[
\hat{y} = \beta_0 + \beta_1 x
\]

where:

- `β₀` = intercept
- `β₁` = coefficient / slope
- `ŷ` = predicted target
- `x` = input feature

## 3. Regression vs Classification

The Titanic dataset contains both kinds of variables, but the **target determines the task**.

| Target | Type | Suitable family |
|---|---|---|
| `Survived` = 0 or 1 | Classification | Logistic Regression, KNN, Trees, etc. |
| `Fare` = continuous value | Regression | Linear Regression, Ridge, Lasso, etc. |

So for this notebook:

```text
X → Passenger-related features
y → Fare
```

We will deliberately leave `Survived` out of the predictor set so the exercise remains a regression problem rather than a disguised classification model.

## 4. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)

## 5. Load the Final Feature-Engineered Dataset

The notebook is designed to work with the project repository layout. It checks a few common locations so the code is less fragile when the notebook is opened from VS Code or Jupyter.

In [ ]:
from pathlib import Path

FILE_NAME = "06_titanic_feature_engineered.csv"

candidate_paths = [
    Path(FILE_NAME),
    Path("../../../03-Data-Preprocessing/Dataset") / FILE_NAME,
    Path("../../../../03-Data-Preprocessing/Dataset") / FILE_NAME,
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find {FILE_NAME}. Place it in the dataset folder or beside the notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
df.head()

## 6. Inspect the Dataset

Before modeling, check shape, data types, and missing values. This is especially important because preprocessing happened in earlier sections of the repository.

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

### A Note About `Sqrt_Fare`

The final feature-engineered file contains `Sqrt_Fare`, but the earlier transformation notebook created it using the already-scaled `Fare`. Since scaled `Fare` can contain negative values, the square-root operation produced invalid values for some rows.

For this notebook, we will **not use `Sqrt_Fare`**. More importantly, both `Log_Fare` and `Sqrt_Fare` are derived directly from the target `Fare`, so using them as predictors would also create **target leakage**.

## 7. Choose the Regression Target

We will predict:

```text
y = Fare
```

The target is continuous. It is already scaled in this dataset, so the values are standardized/transformed rather than original ticket-price amounts.

In [ ]:
target = "Fare"

print("Target:", target)
print("Target dtype:", df[target].dtype)
print("Target range:", (df[target].min(), df[target].max()))

## 8. Build the Feature Matrix

We start with a deliberately selected set of passenger features.

We exclude:

- `Fare` → this is the target
- `Survived` → classification target, not a predictor for this exercise
- `Log_Fare` → directly derived from `Fare`
- `Sqrt_Fare` → directly derived from `Fare` and contains invalid values
- `Age_Group_*` → derived representations of `Age`; they are not needed for this first regression notebook

We also avoid using every rare `Title_*` dummy so that the first model remains easier to interpret.

In [ ]:
feature_columns = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Sex_male",
    "Embarked_Q",
    "Embarked_S",
    "TicketGroupSize",
]

X = df[feature_columns].copy()
y = df[target].copy()

print("Features:")
print(X.columns.tolist())
print("\nTarget:", y.name)
print("\nMissing values in X:")
print(X.isnull().sum()[X.isnull().sum() > 0])

## 9. Simple Linear Regression

Simple Linear Regression uses **one feature** to predict the target:

\[
\hat{y} = \beta_0 + \beta_1x
\]

For a first demonstration, use `Pclass` as the single input feature. This gives us an easy way to visualize the fitted line.

In [ ]:
X_simple = df[["Pclass"]]
y_simple = df[target]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple,
    y_simple,
    test_size=0.20,
    random_state=42,
)

print("Training rows:", len(X_train_s))
print("Testing rows:", len(X_test_s))

## 10. Train Simple Linear Regression with Scikit-Learn

In [ ]:
simple_model = LinearRegression()
simple_model.fit(X_train_s, y_train_s)

print("Intercept (β₀):", simple_model.intercept_)
print("Pclass coefficient (β₁):", simple_model.coef_[0])

### Interpret the Coefficient

The coefficient represents the model's estimated change in predicted scaled `Fare` for a one-unit change in `Pclass`, holding this simple model's structure fixed.

Because `Pclass` in this final file has already been encoded/scaled, the coefficient should **not** be interpreted as “₹ change per passenger class”. It is a change in the dataset's scaled `Fare` with respect to the transformed `Pclass` value.

## 11. Visualize the Fitted Regression Line

In [ ]:
y_pred_simple = simple_model.predict(X_test_s)

plt.figure(figsize=(8, 5))
plt.scatter(X_test_s["Pclass"], y_test_s, alpha=0.6, label="Actual")
plt.scatter(X_test_s["Pclass"], y_pred_simple, alpha=0.6, label="Predicted")

line_x = np.linspace(X_simple["Pclass"].min(), X_simple["Pclass"].max(), 100).reshape(-1, 1)
line_y = simple_model.predict(line_x)
plt.plot(line_x, line_y, linewidth=2, label="Regression line")

plt.title("Simple Linear Regression: Pclass → Fare")
plt.xlabel("Pclass (transformed)")
plt.ylabel("Fare (scaled)")
plt.legend()
plt.tight_layout()
plt.show()

## 12. The Cost Function — Mean Squared Error

Linear Regression commonly minimizes **Mean Squared Error (MSE)**:

\[
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2
\]

The steps are:

```text
Actual value
      ↓
Prediction
      ↓
Error = actual - predicted
      ↓
Square the error
      ↓
Average all squared errors
      ↓
MSE
```

Squaring makes large errors more costly and keeps positive and negative errors from cancelling each other out.

In [ ]:
simple_mse = mean_squared_error(y_test_s, y_pred_simple)
print("Simple model MSE:", simple_mse)

## 13. Gradient Descent — Intuition

One way to fit a linear model is to repeatedly update the parameters in the direction that reduces the cost function.

Conceptually:

```text
Start with parameters
        ↓
Calculate predictions
        ↓
Calculate cost
        ↓
Calculate gradients
        ↓
Update parameters
        ↓
Repeat
```

The learning rate `α` controls how large each update is.

\[
\beta_j \leftarrow \beta_j - \alpha \frac{\partial J}{\partial \beta_j}
\]

Scikit-learn's `LinearRegression` uses an efficient closed-form least-squares solver rather than the educational gradient-descent loop shown below.

## 14. Simple Linear Regression from Scratch

For one feature, the least-squares solution can be written directly using the data. This helps connect the formula to the library implementation.

In [ ]:
x = X_train_s["Pclass"].to_numpy(dtype=float)
y_train_array = y_train_s.to_numpy(dtype=float)

# Closed-form slope and intercept for simple linear regression.
x_mean = x.mean()
y_mean = y_train_array.mean()

beta_1 = np.sum((x - x_mean) * (y_train_array - y_mean)) / np.sum((x - x_mean) ** 2)
beta_0 = y_mean - beta_1 * x_mean

print("Scratch intercept:", beta_0)
print("Scratch coefficient:", beta_1)

# Predict the test set.
y_pred_scratch = beta_0 + beta_1 * X_test_s["Pclass"].to_numpy(dtype=float)

print("Scratch MSE:", mean_squared_error(y_test_s, y_pred_scratch))

The scratch implementation should give essentially the same coefficients and predictions as scikit-learn, apart from small floating-point differences.

That is the important connection:

```text
Mathematical least squares
          ≈
scikit-learn LinearRegression
```

## 15. Multiple Linear Regression

Simple Linear Regression uses one predictor. **Multiple Linear Regression** uses several predictors:

\[
\hat{y} = \beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p
\]

Here, we use the selected passenger features to predict scaled `Fare`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

model = LinearRegression()
model.fit(X_train, y_train)

print("Intercept:", model.intercept_)

coefficients = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_,
}).sort_values("Coefficient", key=lambda s: s.abs(), ascending=False)

coefficients

### Reading the Coefficients

A coefficient describes the model's estimated change in predicted target for a one-unit change in that feature, **assuming the other included features stay fixed**.

Coefficient size alone is not always enough to decide which feature is “most important”: units, scaling, feature correlations, and model specification all matter.

## 16. Predictions

In [ ]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Actual_Fare": y_test.values,
    "Predicted_Fare": y_pred,
})

results.head(10)

## 17. Regression Metrics

We will use three common metrics.

### Mean Absolute Error (MAE)

\[
MAE = \frac{1}{n}\sum |y_i - \hat{y}_i|
\]

Lower is better. It measures the average absolute prediction error.

### Mean Squared Error (MSE)

\[
MSE = \frac{1}{n}\sum (y_i - \hat{y}_i)^2
\]

Lower is better. Large errors receive more penalty.

### Root Mean Squared Error (RMSE)

\[
RMSE = \sqrt{MSE}
\]

Lower is better. It has the same unit as the target.

### R² Score

R² measures the proportion of variance in the target explained by the linear model. Higher is better, with `1.0` representing a perfect fit. R² can also be negative for a model that performs worse than the baseline mean prediction.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R²"],
    "Value": [mae, mse, rmse, r2],
})

metrics

## 18. Compare Simple vs Multiple Linear Regression

In [ ]:
simple_metrics = {
    "Model": "Simple Linear Regression",
    "MAE": mean_absolute_error(y_test_s, y_pred_simple),
    "MSE": mean_squared_error(y_test_s, y_pred_simple),
    "RMSE": np.sqrt(mean_squared_error(y_test_s, y_pred_simple)),
    "R²": r2_score(y_test_s, y_pred_simple),
}

multiple_metrics = {
    "Model": "Multiple Linear Regression",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R²": r2,
}

comparison = pd.DataFrame([simple_metrics, multiple_metrics])
comparison

## 19. Residual Analysis

A **residual** is:

\[
Residual = y - \hat{y}
\]

Residual analysis helps us check whether the linear relationship is a reasonable approximation. Ideally, residuals should not show a strong systematic pattern.

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(0, linewidth=2)
plt.title("Residuals vs Predicted Fare")
plt.xlabel("Predicted Fare (scaled)")
plt.ylabel("Residual")
plt.tight_layout()
plt.show()

## 20. Actual vs Predicted Values

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.6)

line_min = min(y_test.min(), y_pred.min())
line_max = max(y_test.max(), y_pred.max())
plt.plot([line_min, line_max], [line_min, line_max], linewidth=2)

plt.title("Actual vs Predicted Fare")
plt.xlabel("Actual Fare (scaled)")
plt.ylabel("Predicted Fare (scaled)")
plt.tight_layout()
plt.show()

## 21. Important Assumptions of Linear Regression

Linear Regression works best when its assumptions are reasonably satisfied.

### 1. Linearity
The relationship between predictors and target should be approximately linear.

### 2. Independence
Observations should not be strongly dependent on one another.

### 3. Homoscedasticity
The spread of residuals should be reasonably consistent across predicted values.

### 4. Low multicollinearity
Predictors should not be excessively redundant with one another. Strong multicollinearity can make coefficient estimates unstable.

### 5. Residual distribution
For classical statistical inference, residuals are often assumed to be approximately normally distributed. Prediction does not always require perfect normality.

## 22. Why This Titanic Example Has Limitations

This notebook is intentionally educational. There are several reasons not to treat the resulting model as a serious fare-pricing system:

- `Fare` has already been scaled in the source dataset.
- The Titanic dataset is small and historical.
- Some predictors are engineered representations rather than raw business variables.
- Relationships between passenger characteristics and fare are not necessarily well described by a straight line.
- Several variables can be correlated, which affects coefficient interpretation.

The purpose here is to learn **how Linear Regression works**, not to claim that this is the best way to predict Titanic fares.

## 23. Key Takeaways

- Linear Regression predicts a continuous numerical target.
- `Survived` is a classification target, so it is intentionally not used as the regression target here.
- We used `Fare` as a continuous target from the final Titanic feature-engineered file.
- Simple Linear Regression uses one predictor; Multiple Linear Regression uses several.
- The model learns an intercept and coefficients.
- MSE is a common optimization objective; MAE, MSE, RMSE, and R² are useful evaluation metrics.
- Residual analysis helps check whether a linear model is a reasonable approximation.
- Features derived from the target must not be used as predictors because they create target leakage.
- The next regression topics in this repository are **Polynomial Regression**, then **Ridge** and **Lasso**.